<a href="https://colab.research.google.com/github/Mungad/Refined-English-to-Swahili-translation-pipeline/blob/main/English_Swahili_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Evaluating English-to-Swahili Text Translation

In [1]:
!pip install "datasets==2.19.0" evaluate sacrebleu torch transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 23.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


## Cleaned Workspace

In [2]:
import pandas as pd
import torch
import evaluate
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm

# 1. Load Dataset
dataset = load_dataset("facebook/flores", "eng_Latn-swh_Latn", trust_remote_code=True)
test_data = dataset['dev'].select(range(50))
english_texts = test_data['sentence_eng_Latn']
reference_swahili_texts = test_data['sentence_swh_Latn']

# 2. Load Metrics
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

# 3. Load Model and Tokenizer
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
target_lang_id = tokenizer.convert_tokens_to_ids("swh_Latn")

print(f"Setup complete. Model loaded on {device}.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Setup complete. Model loaded on cuda.


In [3]:
import pandas as pd
from tqdm import tqdm

# Load the official devtest split
full_test_data = dataset['devtest']
eng_full = full_test_data['sentence_eng_Latn']
swh_full = full_test_data['sentence_swh_Latn']

print(f"Starting translation of {len(eng_full)} sentences...")

all_predictions = []

for text in tqdm(eng_full, desc="Translating"):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=target_lang_id,
        max_length=150
    )
    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    all_predictions.append(translated_text)

# Calculate final scores
references_full = [[ref] for ref in swh_full]
final_bleu = sacrebleu.compute(predictions=all_predictions, references=references_full)

print(f"\nFull DevTest BLEU Score: {final_bleu['score']:.2f}")

# Save to CSV
results_df = pd.DataFrame({
    'English_Source': eng_full,
    'Swahili_Reference': swh_full,
    'Model_Prediction': all_predictions
})
results_df.to_csv('translation_results.csv', index=False)
print("Results saved to translation_results.csv")

Starting translation of 1012 sentences...



Translating: 100%|██████████| 1012/1012 [10:38<00:00,  1.58it/s]


Full DevTest BLEU Score: 32.00
Results saved to translation_results.csv


In [4]:
# Updated glossary to include more phrasing variations
tech_glossary = {
    "Akili ya bandia": "Akili unde",
    "akili ya bandia": "akili unde",
    "Akili bandia": "Akili unde",
    "akili bandia": "akili unde",
    "Kujifunza kwa mashine": "Ujifunzaji mashine",
    "Kompyuta ya mawingu": "Ukokotoaji wa wingu",
    "Sayansi ya habari": "Sayansi ya data",
    "Ukweli halisi": "Uhalisia pepe",
    "Usalama wa kompyuta": "Usalama wa kimtandao"
}

import pandas as pd
df = pd.read_csv('translation_results.csv')

for wrong_term, correct_term in tech_glossary.items():
    df['Model_Prediction'] = df['Model_Prediction'].str.replace(wrong_term, correct_term, regex=False)

df.to_csv('improved_translation_results.csv', index=False)
print("Glossary updated. CSV has been re-processed with 'Akili unde' corrections.")

Glossary updated. CSV has been re-processed with 'Akili unde' corrections.


In [5]:
from bs4 import BeautifulSoup
import pandas as pd
import os

# 1. Scrape local HTML file
html_path = '/content/index.html'
if os.path.exists(html_path):
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    # Extract unique text from common tags
    tags = ['h1', 'h2', 'h3', 'p', 'li', 'a', 'span']
    extracted_text = list(set([el.get_text().strip() for el in soup.find_all(tags) if len(el.get_text().strip()) > 2]))

    # Save to initial source CSV
    source_df = pd.DataFrame({'English_Source': extracted_text})
    source_df.to_csv('website_source_text.csv', index=False)
    print(f"Extracted {len(extracted_text)} phrases to website_source_text.csv")
else:
    print(f"Error: {html_path} not found in storage.")



Error: /content/index.html not found in storage.


In [9]:
# 2. Translate and create Key-Value Pair CSV
if 'extracted_text' in globals() and len(extracted_text) > 0:
    print(f"Starting translation of {len(extracted_text)} extracted phrases...")
    results = []

    for text in extracted_text:
        # Generate translation with refined parameters
        inputs = tokenizer(text, return_tensors="pt").to(device)
        tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=200,
            num_beams=5,
            repetition_penalty=2.5
        )
        raw_output = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

        # Apply Technical Glossary
        refined_output = raw_output
        for wrong, correct in tech_glossary.items():
            refined_output = refined_output.replace(wrong, correct)

        results.append({'English': text, 'Swahili': refined_output})

    # Save final key-value pairs to CSV
    final_df = pd.DataFrame(results)
    final_df.to_csv('final_website_translations.csv', index=False)
    print("SUCCESS: final_website_translations.csv has been created.")
    display(final_df.head())
else:
    print("Error: No text found to translate. Please ensure the scraping step in the previous cell completed successfully.")

Error: No text found to translate. Please ensure the scraping step in the previous cell completed successfully.


Translate Your Own Custom Text
Now for the fun part! Let's wrap our translation logic into a clean, reusable Python function. This will let you type in any English sentence and instantly get the Swahili output.

Updating the dictionary

In [ ]:
def test_model_with_glossary(english_query):
    inputs = tokenizer(english_query, return_tensors="pt").to(device)
    tokens = model.generate(
        **inputs,
        forced_bos_token_id=target_lang_id,
        max_length=150,
        num_beams=5,
        repetition_penalty=2.5
    )
    raw_output = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

    refined_output = raw_output
    for wrong, correct in tech_glossary.items():
        refined_output = refined_output.replace(wrong, correct)

    print(f"Input: {english_query}")
    print(f"Model (Raw): {raw_output}")
    print(f"Refined (Glossary): {refined_output}")

test_model_with_glossary("ten")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Expanded Custom Dictionary for Modern Swahili Tech Terms
# Add both capitalized and lowercase versions to catch different sentence placements
tech_glossary = {
    # Artificial Intelligence
    "Akili ya bandia": "Akili unde",
    "akili ya bandia": "akili unde",
    "Akili ya Bandia": "Akili unde",

    # Machine Learning
    "Kujifunza kwa mashine": "Ujifunzaji mashine",
    "kujifunza kwa mashine": "ujifunzaji mashine",

    # Cloud Computing
    "Kompyuta ya mawingu": "Ukokotoaji wa wingu",
    "kompyuta ya mawingu": "ukokotoaji wa wingu",

    # Data Science
    "Sayansi ya habari": "Sayansi ya data", # Models sometimes confuse 'data' with 'habari' (news)
    "sayansi ya habari": "sayansi ya data",

    # Virtual Reality
    "Ukweli halisi": "Uhalisia pepe",
    "ukweli halisi": "uhalisia pepe",

    # Cybersecurity
    "Usalama wa kompyuta": "Usalama wa kimtandao",
    "usalama wa kompyuta": "usalama wa kimtandao"
}

# 1. Create the UI Elements
text_input = widgets.Textarea(
    value='',
    placeholder='Type English tech sentence here (e.g., Cloud computing and machine learning are growing)...',
    description='English:',
    layout=widgets.Layout(width='80%', height='100px')
)

translate_button = widgets.Button(
    description='Translate to Swahili',
    button_style='success',
    icon='language'
)

output_area = widgets.Output()

# 2. Define the translation and correction logic
def on_translate_clicked(b):
    with output_area:
        output_area.clear_output()
        text = text_input.value

        if not text.strip():
            print("Please enter some text to translate.")
            return

        print("Translating via NLLB...")

        # Translate using the model
        inputs = tokenizer(text, return_tensors="pt").to(device)
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=150
        )
        raw_translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

        # Apply the expanded glossary corrections
        improved_translation = raw_translation
        for wrong_term, correct_term in tech_glossary.items():
            improved_translation = improved_translation.replace(wrong_term, correct_term)

        print("-" * 50)
        print(f"🤖 Raw Model Output:   {raw_translation}")
        print(f"✅ Polished Swahili:   {improved_translation}")
        print("-" * 50)

# 3. Link and Display
translate_button.on_click(on_translate_clicked)

print("✨ Advanced Swahili Tech Translator ✨")
display(text_input, translate_button, output_area)

In [ ]:
print("--- Improved Swahili Tech Translator Loop ---")
print("Type your English text to translate. Type 'quit' to stop.\n")

while True:
    user_input = input("English: ")

    if user_input.lower().strip() == 'quit':
        print("Exiting... Kwaheri!")
        break

    if not user_input.strip():
        continue

    # 1. Generate Translation with improved parameters
    inputs = tokenizer(user_input, return_tensors="pt").to(device)
    tokens = model.generate(
        **inputs,
        forced_bos_token_id=target_lang_id,
        max_length=150,
        num_beams=5,
        repetition_penalty=2.5,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    raw_output = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

    # 2. Apply Updated Glossary Corrections
    refined_output = raw_output
    for wrong, correct in tech_glossary.items():
        refined_output = refined_output.replace(wrong, correct)

    print(f"Raw Model: {raw_output}")
    print(f"Refined:   {refined_output}")
    print("-" * 30)

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
from tqdm import tqdm

def translate_website(url, output_file):
    print(f"Scraping content from {url}...")
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract text from common tags (headings, paragraphs, list items)
    text_elements = soup.find_all(['h1', 'h2', 'h3', 'p', 'li'])
    unique_sentences = list(set([el.get_text().strip() for el in text_elements if len(el.get_text().strip()) > 5]))

    print(f"Found {len(unique_sentences)} unique sentences. Starting translation...")

    site_translations = {}

    for text in tqdm(unique_sentences):
        # 1. Generate Raw Translation with refined params
        inputs = tokenizer(text, return_tensors="pt").to(device)
        tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=200,
            num_beams=5,
            repetition_penalty=2.5
        )
        raw_output = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

        # 2. Apply Technical Glossary
        refined_output = raw_output
        for wrong, correct in tech_glossary.items():
            refined_output = refined_output.replace(wrong, correct)

        site_translations[text] = refined_output

    # 3. Save to JSON
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(site_translations, f, ensure_ascii=False, indent=4)

    print(f"\nSUCCESS: {output_file} has been created with {len(site_translations)} translations.")

# Execute the translation for the DSAIL site
translate_website("https://dekut-dsail.github.io/", "dsail_site_swahili.json")

In [ ]:
import pandas as pd

# 1. Update the glossary with specific user corrections
corrections = {
    "Emerson Munene": "Emerson Munene",
    "Team": "Timu",
    "Gallery": "Ghala la picha",
    "About us": "Kutuhusu",
    "Resources": "Rasilimali",
    "Blogs": "Blogu",
    "Space Summer Camp 2025": "Kambi ya Nafasi ya Majira ya joto 2025",
    "Datasets": "Seti za data"
}

# Update the main tech_glossary for future use
tech_glossary.update(corrections)

# 2. Load the existing final translations
final_df = pd.read_csv('final_website_translations.csv')

# 3. Apply the specific corrections to the Swahili column
def apply_strict_corrections(text):
    for eng, swh in corrections.items():
        # If the English source matches exactly, or if the bad translation is found, replace it
        # This also handles the 'gallery gallery' style repetitions by replacing the whole string
        if text.lower().strip() == eng.lower().strip() or eng in text:
            # For specific mapping provided by user:
            if eng == "Team": return "Timu"
            if eng == "Gallery": return "Ghala la picha"
            if eng == "About us": return "Kutuhusu"
            if eng == "Resources": return "Rasilimali"
            if eng == "Blogs": return "Blogi"
            if eng == "Space Summer Camp 2025": return "Kambi ya Nafasi ya Majira ya joto 2025"
            if eng == "Datasets": return "Seti za data"
    return text

# Apply corrections to the DataFrame
# We check the 'English' column to decide what the 'Swahili' column should be
for index, row in final_df.iterrows():
    eng_val = str(row['English']).strip()
    if eng_val in corrections:
        final_df.at[index, 'Swahili'] = corrections[eng_val]

# Save the corrected version
final_df.to_csv('final_website_translations.csv', index=False)
print("Corrections applied to final_website_translations.csv")
display(final_df.head(20))

In [ ]:
import pandas as pd
import json

# Load your file
df = pd.read_csv('improved_translation_results.csv')

# Create a dictionary mapping English to Swahili (using Swahili_Reference)
# You can change 'Swahili_Reference' to 'Model_Prediction' if preferred
translation_map = dict(zip(df['English_Source'], df['Swahili_Reference']))

# Save as a JSON file for your website
with open('swahili_translations.json', 'w', encoding='utf-8') as f:
    json.dump(translation_map, f, ensure_ascii=False, indent=4)

# **Part 2: Moving the Glossary to a JSON File**
Hardcoding a dictionary is fine for testing, but in production, you want a separate file. This allows non-programmers (like translators or linguists) to update the vocabulary without touching your Python code.

1. Create the JSON file

Convert Your CSV to a Website-Ready JSON
Websites usually need translations in a specific "Key-Value" format, where the English text is the key, and the Swahili text is the value (e.g., {"Hello": "Hujambo"}).

In [ ]:
import json

df = pd.read_csv('improved_translation_results.csv')
website_dictionary = dict(zip(df['English_Source'], df['Model_Prediction']))

with open('swahili_translations.json', 'w', encoding='utf-8') as f:
    json.dump(website_dictionary, f, ensure_ascii=False, indent=4)

print("SUCCESS: swahili_translations.json is ready for download.")

In [ ]:
# Check if the results file exists from a previous run to avoid re-translating 1000+ sentences
import os
if os.path.exists('improved_translation_results.csv'):
    df_check = pd.read_csv('improved_translation_results.csv')
    display(df_check.head())
    print("Found existing improved results file.")
else:
    print("Results file not found. Ready to run full evaluation if needed.")